# 09 - Options Dashboard

One view: everything about the current ETH options market.

Combines: PCR, ATM IV, skew, max pain, spot, trend.


## Setup

In [ ]:
import sys, sqlite3
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / "database").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

OPT_DB = _p / "database" / "options.db"
MKT_DB = _p / "database" / "market.db"

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def opt(q, **p):
    with sqlite3.connect(OPT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

def mkt(q, **p):
    with sqlite3.connect(MKT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

print("options.db:", OPT_DB)
print("market.db :", MKT_DB)

## 1. Load data

In [ ]:
s = opt("SELECT * FROM option_summary WHERE underlying='ETH' ORDER BY ts")
oc = opt("SELECT underlying_px FROM option_chain ORDER BY ts DESC LIMIT 1")
iv = opt("""
    SELECT * FROM implied_volatility
    WHERE underlying='ETH'
      AND ts = (SELECT MAX(ts) FROM implied_volatility WHERE underlying='ETH')
""")
print(f"summary rows: {len(s)}")
print(f"iv rows     : {len(iv)}")

## 2. Snapshot

In [ ]:
if not s.empty:
    last = s.iloc[-1]
    spot = float(oc.underlying_px.iloc[0]) if not oc.empty and oc.underlying_px.iloc[0] else None

    print("=" * 55)
    print("  ETH OPTIONS DASHBOARD")
    print("=" * 55)
    print(f"  as of       : {pd.to_datetime(last.ts, unit='ms')}")
    if spot: print(f"  spot        : {spot:.2f}")
    print(f"  PCR (OI)    : {last.pcr_oi}")
    print(f"  PCR (vol)   : {last.pcr_vol}")
    print(f"  max pain    : {last.max_pain}")
    print(f"  put OI max  : {last.put_oi_max_strike}")
    print(f"  call OI max : {last.call_oi_max_strike}")
    print(f"  ATM IV      : {last.atm_iv}")
    print(f"  25d skew    : {last.iv_25d_skew}")
    print(f"  term struct : {last.term_structure}")
    print("=" * 55)

## 3. Signals

In [ ]:
if not s.empty:
    last = s.iloc[-1]
    signals = []

    # PCR
    if last.pcr_oi and last.pcr_oi > 1.3:
        signals.append(("PCR", "bearish", "put-heavy"))
    elif last.pcr_oi and last.pcr_oi < 0.7:
        signals.append(("PCR", "bullish", "call-heavy"))
    else:
        signals.append(("PCR", "neutral", ""))

    # skew
    if last.iv_25d_skew and last.iv_25d_skew > 0.05:
        signals.append(("skew", "bearish", "puts expensive"))
    elif last.iv_25d_skew and last.iv_25d_skew < -0.02:
        signals.append(("skew", "bullish", "calls expensive"))
    else:
        signals.append(("skew", "neutral", ""))

    # max pain vs spot
    if last.max_pain and spot:
        dist = (last.max_pain - spot) / spot * 100
        if dist > 3:
            signals.append(("max pain", "bullish", f"+{dist:.1f}% above spot"))
        elif dist < -3:
            signals.append(("max pain", "bearish", f"{dist:.1f}% below spot"))
        else:
            signals.append(("max pain", "neutral", f"{dist:+.1f}%"))

    for name, sig, note in signals:
        print(f"  {name:<12} {sig:<10} {note}")

## 4. Composite

In [ ]:
if not s.empty:
    bull = sum(1 for _, sig, _ in signals if sig == "bullish")
    bear = sum(1 for _, sig, _ in signals if sig == "bearish")
    neut = sum(1 for _, sig, _ in signals if sig == "neutral")

    print(f"bullish: {bull}")
    print(f"bearish: {bear}")
    print(f"neutral: {neut}")

    if bull > bear:   verdict = "NET BULLISH"
    elif bear > bull: verdict = "NET BEARISH"
    else:             verdict = "NEUTRAL"

    print()
    print(f"  >>> {verdict}  <<<")